In [16]:
from pathlib import Path

from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break


In [17]:
# Check if the Databricks session is already created
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
df = spark.range(3)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



In [ ]:
# Read the data from the delta lake
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()
path = "/FileStore/bronze/"
rows = [
    {"path": e.path, "is_dir": e.is_dir, "size": e.file_size}
    for e in w.dbfs.list(path)
]
pd.DataFrame(rows)


,path,is_dir,size
0,/FileStore/bronze/github,True,0
1,/FileStore/bronze/pypi,True,0
2,/FileStore/bronze/stackoverflow,True,0


In [13]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ddc_databricks.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ddc_databricks.gold")

DataFrame[]

In [14]:
# Repos
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/github/repos/")
df.write.format("delta").mode("overwrite").saveAsTable("ddc_databricks.bronze.github_repos")

# Contributors
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/github/contributors/")
df.write.format("delta").mode("overwrite").saveAsTable("ddc_databricks.bronze.github_contributors")

# Readmes
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/github/readmes/")
df.write.format("delta").mode("overwrite").saveAsTable("ddc_databricks.bronze.github_readmes")

# Events
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/github/events/")
df.write.format("delta").mode("overwrite").saveAsTable("ddc_databricks.bronze.github_events")

In [25]:
from pyspark.sql import functions as F

# load pypi metadat
df_raw = spark.read.format("binaryFile").load("dbfs:/FileStore/bronze/pypi/metadata/")

df = df_raw.select(
    F.get_json_object(F.decode("content", "utf-8"), "$._ingested_at").alias("_ingested_at"),
    F.get_json_object(F.decode("content", "utf-8"), "$._source").alias("_source"),
    F.get_json_object(F.decode("content", "utf-8"), "$._pypi_package").alias("_pypi_package"),
    F.get_json_object(F.decode("content", "utf-8"), "$._endpoint").alias("_endpoint"),
    F.get_json_object(F.decode("content", "utf-8"), "$.data").alias("data"),
)

# Verify first
df.select("_pypi_package", "_ingested_at").show(5)

+-------------+--------------------+
|_pypi_package|        _ingested_at|
+-------------+--------------------+
|        numpy|2026-03-24T10:30:...|
|       pandas|2026-03-24T10:30:...|
| scikit-learn|2026-03-24T10:30:...|
|        torch|2026-03-24T10:31:...|
|       django|2026-03-24T10:30:...|
+-------------+--------------------+
only showing top 5 rows


In [26]:
# save pypi metadata in deltalake
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.pypi_metadata")

In [10]:
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/pypi/downloads_overall/")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.pypi_downloads_overall")
print("pypi downloads overall saved")

pypi downloads overall saved


In [12]:
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/pypi/downloads_recent/")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.pypi_downloads_recent")
print("pypi downloads recent saved")

pypi downloads recent saved


In [28]:
# Storing Stack Overflow Questions in Delta Lake
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/stackoverflow/questions/")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.so_questions")
print("StackOverflow Questions Saved")



StackOverflow Questions Saved


In [32]:
# Storing Stack Overflow Answers in Delta Lake
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/stackoverflow/answers/")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.so_answers")
print("StackOverflow Answers Saved")


StackOverflow Answers Saved


In [30]:
# Storing Stack Overflow Questions in Delta Lake
df = spark.read.option("multiline", "true").json("dbfs:/FileStore/bronze/stackoverflow/tag_info/")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ddc_databricks.bronze.so_tag_info")
print("StackOverflow Tag Information Saved")


StackOverflow Tag Information Saved
